# Module 6 — Statistical Tests for A/B Testing

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Intermediate  
> **Runtime**: ~4 minutes  
> **Key Libraries**: SciPy, Statsmodels, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 1 (Optimized Pandas)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic A/B Test Data Generation](#3-synthetic-ab-test-data-generation)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Frequentist Tests](#5-frequentist-tests)
   - 5a. Two-Sample T-Test
   - 5b. Mann-Whitney U Test
   - 5c. Chi-Squared Test
   - 5d. Z-Test for Proportions
6. [Bootstrap Confidence Intervals](#6-bootstrap-confidence-intervals)
7. [Power Analysis & Sample Size](#7-power-analysis--sample-size)
8. [Visualization Dashboard](#8-visualization-dashboard)
9. [Key Business Takeaway](#9-key-business-takeaway)

---
## §1 · Business Context

### Why does A/B testing matter at Revolut?

Revolut runs **hundreds of A/B tests per month** across product, marketing, and risk:

| Team | Experiment | Metric |
|------|-----------|--------|
| **Product** | New "Send Money" button layout | Conversion rate (%) |
| **Marketing** | Push notification copy A vs. B | Click-through rate (%) |
| **Risk** | New fraud-scoring threshold | False-positive rate (%) |
| **Pricing** | Premium plan price £7.99 vs. £9.99 | Subscription uptake (%) |

**The stakes**: a false-positive (declaring a winner when there is none) costs millions in
engineering time and lost revenue. A false-negative (missing a real winner) leaves money on the table.

In this notebook we will:
1. Generate a realistic 50/50 split-test dataset (20K users).
2. Run four frequentist tests: t-test, Mann-Whitney, Chi-Squared, Z-test.
3. Compute bootstrap confidence intervals.
4. Perform power analysis to determine minimum sample size.
5. Build an executive-ready results dashboard.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Statistics ───────────────────────────────────────────────────
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import TTestIndPower, NormalIndPower

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", palette="muted")

print("✓ All imports loaded")

---
## §3 · Synthetic A/B Test Data Generation

### Scenario
Revolut is testing a **new "Send Money" button** (Variant B) vs. the current design (Variant A).

**Primary metrics**:
- `conversion`: Did the user complete a transfer? (binary)
- `transfer_amount`: How much did they transfer? (continuous)
- `time_to_complete`: Seconds from click to completion (continuous)

We inject a **true 5% relative lift** in conversion for Variant B.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_PER_GROUP = 10_000   # 10K users per variant
CONTROL_CVR = 0.120    # 12% baseline conversion rate
TREATMENT_CVR = 0.126  # 5% relative lift (12.6%)

print(f"Generating A/B test: {N_PER_GROUP:,} users per group")
print(f"  Control CVR  : {CONTROL_CVR * 100:.1f}%")
print(f"  Treatment CVR: {TREATMENT_CVR * 100:.1f}%  (true lift: +{(TREATMENT_CVR / CONTROL_CVR - 1) * 100:.1f}%)")

# ── Control group (A) ────────────────────────────────────────────
control = pd.DataFrame({
    "user_id":         np.arange(1, N_PER_GROUP + 1),
    "variant":         "A",
    "conversion":      np.random.binomial(1, CONTROL_CVR, N_PER_GROUP),
    "transfer_amount": np.where(
        np.random.random(N_PER_GROUP) < CONTROL_CVR,
        np.random.lognormal(mean=6.5, sigma=1.0, size=N_PER_GROUP),
        0
    ),
    "time_to_complete": np.where(
        np.random.random(N_PER_GROUP) < CONTROL_CVR,
        np.random.gamma(shape=3, scale=10, size=N_PER_GROUP),
        np.nan
    ),
})

# ── Treatment group (B) ──────────────────────────────────────────
treatment = pd.DataFrame({
    "user_id":         np.arange(N_PER_GROUP + 1, 2 * N_PER_GROUP + 1),
    "variant":         "B",
    "conversion":      np.random.binomial(1, TREATMENT_CVR, N_PER_GROUP),
    "transfer_amount": np.where(
        np.random.random(N_PER_GROUP) < TREATMENT_CVR,
        np.random.lognormal(mean=6.5, sigma=1.0, size=N_PER_GROUP),
        0
    ),
    "time_to_complete": np.where(
        np.random.random(N_PER_GROUP) < TREATMENT_CVR,
        np.random.gamma(shape=3, scale=9.5, size=N_PER_GROUP),  # slightly faster
        np.nan
    ),
})

df = pd.concat([control, treatment], ignore_index=True)
df["transfer_amount"] = np.round(df["transfer_amount"], 2)
df["time_to_complete"] = np.round(df["time_to_complete"], 1)

print(f"\n✓ Shape: {df.shape}")
df.head(10)

---
## §4 · Exploratory Data Analysis

In [ ]:
# Summary statistics by variant
summary = df.groupby("variant").agg(
    users=("user_id", "count"),
    conversions=("conversion", "sum"),
    conversion_rate=("conversion", "mean"),
    avg_transfer=("transfer_amount", "mean"),
    median_transfer=("transfer_amount", "median"),
    avg_time=("time_to_complete", "mean"),
    median_time=("time_to_complete", "median"),
).round(4)

summary["conversion_rate_pct"] = (summary["conversion_rate"] * 100).round(2)
summary

In [ ]:
# Distribution of transfer amounts (converters only)
converters = df[df["conversion"] == 1]

fig = px.histogram(
    converters, x="transfer_amount", color="variant",
    nbins=80, marginal="box",
    title="Transfer Amount Distribution (Converters Only)",
    labels={"transfer_amount": "Transfer Amount (£)"},
    color_discrete_map={"A": "#636EFA", "B": "#EF553B"},
)
fig.update_layout(height=450, barmode="overlay", opacity=0.7)
fig.show()

In [ ]:
# Conversion rate comparison
cvr_a = df[df["variant"] == "A"]["conversion"].mean()
cvr_b = df[df["variant"] == "B"]["conversion"].mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=["Control (A)", "Treatment (B)"],
    y=[cvr_a, cvr_b],
    marker_color=["#636EFA", "#EF553B"],
    text=[f"{cvr_a*100:.2f}%", f"{cvr_b*100:.2f}%"],
    textposition="outside",
))

fig.update_layout(
    title="Observed Conversion Rates",
    yaxis_title="Conversion Rate",
    yaxis_tickformat=".1%",
    height=380,
    yaxis_range=[0, max(cvr_a, cvr_b) * 1.3],
)
fig.show()

observed_lift = (cvr_b / cvr_a - 1) * 100
print(f"Observed lift: {observed_lift:+.2f}%")
print(f"True lift (injected): {(TREATMENT_CVR / CONTROL_CVR - 1) * 100:+.1f}%")
print("\n⚠️  Observed lift may differ from true lift due to sampling noise")

---
## §5 · Frequentist Tests

### 5a — Two-Sample T-Test (Continuous Metric: Transfer Amount)

**Null hypothesis**: Mean transfer amount is the same for A and B.  
**Alternative**: Mean transfer amount differs between A and B.

In [ ]:
# Extract transfer amounts (all users, including zeros for non-converters)
amount_a = df[df["variant"] == "A"]["transfer_amount"].values
amount_b = df[df["variant"] == "B"]["transfer_amount"].values

# Welch's t-test (doesn't assume equal variances)
t_stat, p_value = stats.ttest_ind(amount_a, amount_b, equal_var=False)

print("=" * 60)
print("Welch's Two-Sample T-Test")
print("=" * 60)
print(f"Control mean  : £{amount_a.mean():,.2f}")
print(f"Treatment mean: £{amount_b.mean():,.2f}")
print(f"Difference    : £{amount_b.mean() - amount_a.mean():,.2f}")
print(f"\nT-statistic   : {t_stat:.4f}")
print(f"P-value       : {p_value:.6f}")
print(f"\nDecision (α=0.05): {'Reject H₀ — significant difference' if p_value < 0.05 else 'Fail to reject H₀ — no significant difference'}")

# Effect size (Cohen's d)
pooled_std = np.sqrt((amount_a.std()**2 + amount_b.std()**2) / 2)
cohens_d = (amount_b.mean() - amount_a.mean()) / pooled_std
print(f"\nEffect size (Cohen's d): {cohens_d:.4f}")
if abs(cohens_d) < 0.2:
    print("  → Small effect")
elif abs(cohens_d) < 0.5:
    print("  → Medium effect")
else:
    print("  → Large effect")

### 5b — Mann-Whitney U Test (Non-Parametric Alternative)

Use when data is **not normally distributed** (e.g., transfer amounts are log-normal).

In [ ]:
# Mann-Whitney U test (non-parametric, tests if distributions are equal)
u_stat, p_value_mw = stats.mannwhitneyu(amount_a, amount_b, alternative="two-sided")

print("=" * 60)
print("Mann-Whitney U Test (Non-Parametric)")
print("=" * 60)
print(f"U-statistic   : {u_stat:,.0f}")
print(f"P-value       : {p_value_mw:.6f}")
print(f"\nDecision (α=0.05): {'Reject H₀ — distributions differ' if p_value_mw < 0.05 else 'Fail to reject H₀ — distributions similar'}")

# Compare with t-test
print(f"\nComparison:")
print(f"  T-test p-value       : {p_value:.6f}")
print(f"  Mann-Whitney p-value : {p_value_mw:.6f}")
print("\n💡 Mann-Whitney is more robust to outliers and non-normality")

### 5c — Chi-Squared Test (Categorical: Conversion Rate)

**Null hypothesis**: Conversion rate is independent of variant.  
**Alternative**: Conversion rate depends on variant.

In [ ]:
# Build contingency table
conv_a = df[df["variant"] == "A"]["conversion"].sum()
conv_b = df[df["variant"] == "B"]["conversion"].sum()
no_conv_a = N_PER_GROUP - conv_a
no_conv_b = N_PER_GROUP - conv_b

contingency = np.array([
    [conv_a, no_conv_a],    # Variant A: [converted, not converted]
    [conv_b, no_conv_b],    # Variant B: [converted, not converted]
])

print("Contingency Table:")
print(pd.DataFrame(contingency, 
                   index=["Control (A)", "Treatment (B)"],
                   columns=["Converted", "Not Converted"]))

# Chi-squared test
chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency)

print("\n" + "=" * 60)
print("Chi-Squared Test")
print("=" * 60)
print(f"Chi² statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value       : {p_value_chi2:.6f}")
print(f"\nDecision (α=0.05): {'Reject H₀ — conversion rates differ' if p_value_chi2 < 0.05 else 'Fail to reject H₀ — no significant difference'}")

# Cramér's V (effect size for Chi²)
cramers_v = np.sqrt(chi2_stat / (contingency.sum() * (min(contingency.shape) - 1)))
print(f"\nEffect size (Cramér's V): {cramers_v:.4f}")
print("  → Small effect (< 0.1), Medium (0.1–0.3), Large (> 0.3)")

### 5d — Z-Test for Proportions (Conversion Rate)

More precise than Chi² for comparing two proportions.

In [ ]:
# Z-test for two proportions
conv_counts = np.array([conv_a, conv_b])
n_obs = np.array([N_PER_GROUP, N_PER_GROUP])

z_stat, p_value_z = proportions_ztest(conv_counts, n_obs, alternative="two-sided")

print("=" * 60)
print("Two-Proportion Z-Test")
print("=" * 60)
print(f"Control CVR   : {conv_a / N_PER_GROUP * 100:.2f}%")
print(f"Treatment CVR : {conv_b / N_PER_GROUP * 100:.2f}%")
print(f"Absolute lift : {(conv_b / N_PER_GROUP - conv_a / N_PER_GROUP) * 100:.3f} pp")
print(f"Relative lift : {(conv_b / conv_a - 1) * 100:.2f}%")
print(f"\nZ-statistic   : {z_stat:.4f}")
print(f"P-value       : {p_value_z:.6f}")
print(f"\nDecision (α=0.05): {'Reject H₀ — CVRs differ' if p_value_z < 0.05 else 'Fail to reject H₀ — no significant difference'}")

# Confidence interval for the difference
from statsmodels.stats.proportion import proportion_confint
ci_a = proportion_confint(conv_a, N_PER_GROUP, alpha=0.05, method="wilson")
ci_b = proportion_confint(conv_b, N_PER_GROUP, alpha=0.05, method="wilson")

diff = conv_b / N_PER_GROUP - conv_a / N_PER_GROUP
se_diff = np.sqrt((conv_a / N_PER_GROUP * (1 - conv_a / N_PER_GROUP) / N_PER_GROUP) +
                  (conv_b / N_PER_GROUP * (1 - conv_b / N_PER_GROUP) / N_PER_GROUP))
ci_diff = (diff - 1.96 * se_diff, diff + 1.96 * se_diff)

print(f"\n95% CI for difference: [{ci_diff[0]*100:.3f} pp, {ci_diff[1]*100:.3f} pp]")
if ci_diff[0] > 0 or ci_diff[1] < 0:
    print("  → CI does NOT include 0 → significant difference")
else:
    print("  → CI includes 0 → not significant")

---
## §6 · Bootstrap Confidence Intervals

Bootstrapping resamples the data with replacement to estimate the sampling distribution
of any statistic — no parametric assumptions needed.

In [ ]:
# Bootstrap the difference in conversion rates
def bootstrap_cvr_diff(data_a, data_b, n_bootstrap=10_000, alpha=0.05):
    """Compute bootstrap CI for CVR difference."""
    diffs = []
    n_a, n_b = len(data_a), len(data_b)
    
    for _ in range(n_bootstrap):
        # Resample with replacement
        sample_a = np.random.choice(data_a, size=n_a, replace=True)
        sample_b = np.random.choice(data_b, size=n_b, replace=True)
        diff = sample_b.mean() - sample_a.mean()
        diffs.append(diff)
    
    diffs = np.array(diffs)
    lower = np.percentile(diffs, alpha / 2 * 100)
    upper = np.percentile(diffs, (1 - alpha / 2) * 100)
    se = diffs.std()
    
    return diffs, lower, upper, se

cvr_a_data = df[df["variant"] == "A"]["conversion"].values
cvr_b_data = df[df["variant"] == "B"]["conversion"].values

print("Running 10,000 bootstrap iterations …")
t0 = time.time()
bootstrap_diffs, ci_lower, ci_upper, se_boot = bootstrap_cvr_diff(cvr_a_data, cvr_b_data)
elapsed = time.time() - t0

observed_diff = cvr_b_data.mean() - cvr_a_data.mean()

print(f"✓ Completed in {elapsed:.2f}s")
print(f"\nObserved CVR difference: {observed_diff * 100:.3f} pp")
print(f"Bootstrap SE           : {se_boot * 100:.4f} pp")
print(f"95% Bootstrap CI       : [{ci_lower * 100:.3f} pp, {ci_upper * 100:.3f} pp]")

if ci_lower > 0:
    print("\n✅ CI does NOT include 0 → Treatment B is significantly better")
elif ci_upper < 0:
    print("\n❌ CI does NOT include 0 → Treatment B is significantly worse")
else:
    print("\n⚠️  CI includes 0 → No significant difference detected")

In [ ]:
# Visualize bootstrap distribution
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=bootstrap_diffs * 100,
    nbinsx=80,
    name="Bootstrap Distribution",
    marker_color="#636EFA",
    opacity=0.7,
))

# Add observed difference
fig.add_vline(x=observed_diff * 100, line_dash="solid", line_color="black",
              annotation_text=f"Observed: {observed_diff*100:.3f}pp")

# Add CI
fig.add_vline(x=ci_lower * 100, line_dash="dash", line_color="red",
              annotation_text=f"2.5%: {ci_lower*100:.3f}pp")
fig.add_vline(x=ci_upper * 100, line_dash="dash", line_color="red",
              annotation_text=f"97.5%: {ci_upper*100:.3f}pp")

# Add zero line
fig.add_vline(x=0, line_dash="dot", line_color="gray", annotation_text="No effect (0)")

fig.update_layout(
    title="Bootstrap Distribution of CVR Difference (B - A)",
    xaxis_title="CVR Difference (percentage points)",
    yaxis_title="Frequency",
    height=440,
)
fig.show()

---
## §7 · Power Analysis & Sample Size

### 7.1 — What is Statistical Power?

| Term | Definition |
|------|------------|
| **α (alpha)** | Probability of false positive (Type I error). Typically 0.05. |
| **β (beta)** | Probability of false negative (Type II error). Typically 0.20. |
| **Power (1-β)** | Probability of detecting a true effect. Typically 0.80. |
| **Effect size** | Minimum detectable difference (MDE). |
| **Sample size (N)** | Number of users per group. |

In [ ]:
# Power analysis: How many users do we need?
# Given: baseline CVR = 12%, MDE = 5% relative lift, α = 0.05, power = 0.80

baseline_cvr = 0.12
mde_relative = 0.05  # 5% relative lift
mde_absolute = baseline_cvr * mde_relative

# Use NormalIndPower for proportions
power_analysis = NormalIndPower()

# Calculate required sample size per group
n_required = power_analysis.solve_power(
    effect_size=mde_absolute / np.sqrt(baseline_cvr * (1 - baseline_cvr)),
    alpha=0.05,
    power=0.80,
    alternative="two-sided",
)

print("=" * 60)
print("Power Analysis: Minimum Sample Size")
print("=" * 60)
print(f"Baseline CVR      : {baseline_cvr * 100:.1f}%")
print(f"MDE (relative)    : {mde_relative * 100:.1f}%")
print(f"MDE (absolute)    : {mde_absolute * 100:.3f} pp")
print(f"α (significance)  : 0.05")
print(f"Power (1 - β)     : 0.80")
print(f"\nRequired N per group: {int(np.ceil(n_required)):,}")
print(f"Total users needed : {int(np.ceil(n_required)) * 2:,}")

print(f"\n💡 Our experiment had {N_PER_GROUP:,} per group → {'sufficient' if N_PER_GROUP >= n_required else 'underpowered'}")

### 7.2 — Power Curve: Sample Size vs. Detectable Effect

In [ ]:
# Plot power curve for different sample sizes
sample_sizes = np.arange(1000, 30001, 1000)
mde_values = []

for n in sample_sizes:
    # What MDE can we detect with this sample size?
    effect_size = power_analysis.solve_power(
        effect_size=None,
        nobs1=n,
        alpha=0.05,
        power=0.80,
    )
    mde_abs = effect_size * np.sqrt(baseline_cvr * (1 - baseline_cvr))
    mde_rel = mde_abs / baseline_cvr * 100
    mde_values.append(mde_rel)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample_sizes, y=mde_values,
    mode="lines+markers",
    name="MDE (80% power)",
    line=dict(color="#636EFA", width=2.5),
))

# Add our experiment
fig.add_vline(x=N_PER_GROUP, line_dash="dash", line_color="red",
              annotation_text=f"Current N: {N_PER_GROUP:,}")

fig.update_layout(
    title="Power Curve: Sample Size vs. Minimum Detectable Effect",
    xaxis_title="Sample Size per Group",
    yaxis_title="Minimum Detectable Effect (relative %)",
    height=440,
)
fig.show()

print("💡 Larger samples → smaller effects become detectable")
print("   Diminishing returns after ~20K users per group")

---
## §8 · Visualization Dashboard

In [ ]:
# ── 8.1 Test Results Summary ─────────────────────────────────────
test_results = pd.DataFrame({
    "Test": ["Welch's T-Test", "Mann-Whitney U", "Chi-Squared", "Z-Test (Proportions)", "Bootstrap CI"],
    "Metric": ["Transfer Amount", "Transfer Amount", "Conversion Rate", "Conversion Rate", "Conversion Rate"],
    "Statistic": [f"t={t_stat:.3f}", f"U={u_stat:,.0f}", f"χ²={chi2_stat:.3f}", f"z={z_stat:.3f}", "10K iterations"],
    "P-value": [p_value, p_value_mw, p_value_chi2, p_value_z, "N/A"],
    "Significant?": [p_value < 0.05, p_value_mw < 0.05, p_value_chi2 < 0.05, p_value_z < 0.05, ci_lower > 0],
})

test_results["P-value"] = test_results["P-value"].apply(lambda x: f"{x:.6f}" if isinstance(x, float) else x)
test_results["Significant?"] = test_results["Significant?"].apply(lambda x: "✅ Yes" if x else "❌ No")

print("=" * 80)
print("A/B Test Results Summary")
print("=" * 80)
print(test_results.to_string(index=False))

In [ ]:
# ── 8.2 Conversion Rates with Confidence Intervals ───────────────
fig = go.Figure()

# Control
fig.add_trace(go.Scatter(
    x=["Control (A)"],
    y=[cvr_a * 100],
    error_y=dict(type="data", array=[(ci_a[1] - ci_a[0]) / 2 * 100]),
    mode="markers",
    marker=dict(size=20, color="#636EFA"),
    name="Control",
))

# Treatment
fig.add_trace(go.Scatter(
    x=["Treatment (B)"],
    y=[cvr_b * 100],
    error_y=dict(type="data", array=[(ci_b[1] - ci_b[0]) / 2 * 100]),
    mode="markers",
    marker=dict(size=20, color="#EF553B"),
    name="Treatment",
))

fig.update_layout(
    title="Conversion Rates with 95% Confidence Intervals",
    yaxis_title="Conversion Rate (%)",
    height=400,
    showlegend=False,
)
fig.show()

overlap = (ci_a[1] >= ci_b[0]) and (ci_b[1] >= ci_a[0])
if overlap:
    print("⚠️  CIs overlap — but this doesn't necessarily mean non-significant")
    print("   (Overlapping CIs is a conservative test; use the Z-test instead)")
else:
    print("✅ CIs do NOT overlap → significant difference")

In [ ]:
# ── 8.3 Daily Conversion Rates Over Time ─────────────────────────
# Simulate daily breakdown (assuming experiment ran for 14 days)
df["day"] = pd.to_datetime("2025-01-01") + pd.to_timedelta(df["user_id"] % 14, unit="D")

daily_cvr = df.groupby(["day", "variant"])["conversion"].mean().reset_index()

fig = px.line(
    daily_cvr, x="day", y="conversion", color="variant",
    title="Daily Conversion Rates (A vs. B)",
    labels={"conversion": "CVR", "day": "Date"},
    color_discrete_map={"A": "#636EFA", "B": "#EF553B"},
)
fig.update_yaxes(tickformat=".1%")
fig.update_layout(height=400, legend=dict(orientation="h", y=1.12))
fig.show()

print("💡 Daily fluctuations are normal — always look at the aggregate result")

In [ ]:
# ── 8.4 Executive Summary Card ───────────────────────────────────
print("=" * 70)
print(" " * 20 + "A/B TEST EXECUTIVE SUMMARY")
print("=" * 70)
print(f"\nExperiment: New 'Send Money' Button (B) vs. Current (A)")
print(f"Duration: 14 days  |  Users: {2 * N_PER_GROUP:,} (50/50 split)")
print(f"\nPrimary Metric: Conversion Rate")
print(f"  Control (A)  : {cvr_a * 100:.2f}%")
print(f"  Treatment (B): {cvr_b * 100:.2f}%")
print(f"  Lift         : {(cvr_b / cvr_a - 1) * 100:+.2f}%")
print(f"  P-value      : {p_value_z:.6f}")
print(f"  Significance : {'✅ Statistically significant (α=0.05)' if p_value_z < 0.05 else '❌ Not significant'}")
print(f"  95% CI (diff): [{ci_diff[0]*100:.3f} pp, {ci_diff[1]*100:.3f} pp]")
print(f"\nSecondary Metric: Average Transfer Amount")
print(f"  Control (A)  : £{amount_a.mean():,.2f}")
print(f"  Treatment (B): £{amount_b.mean():,.2f}")
print(f"  P-value      : {p_value:.6f}")
print(f"  Significance : {'✅ Statistically significant' if p_value < 0.05 else '❌ Not significant'}")
print(f"\nRecommendation: {'🚀 ROLLOUT Variant B' if p_value_z < 0.05 and cvr_b > cvr_a else '⏸️  HOLD — no significant improvement'}")
print("=" * 70)

---
## §9 · Key Business Takeaway

> ### 🎯 Action Items for a Data Scientist
>
> | Test | When to Use | Key Function |
> |------|-------------|--------------|
> | **Welch's T-Test** | Continuous metric (e.g., revenue, time) | `scipy.stats.ttest_ind(equal_var=False)` |
> | **Mann-Whitney U** | Non-normal continuous data | `scipy.stats.mannwhitneyu()` |
> | **Chi-Squared** | Categorical (e.g., conversion yes/no) | `scipy.stats.chi2_contingency()` |
> | **Z-Test** | Two proportions | `statsmodels.stats.proportion.proportions_ztest()` |
> | **Bootstrap CI** | Any statistic (no assumptions) | Custom resampling loop |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Always run power analysis BEFORE the experiment** to determine sample size.
>    - Use `statsmodels.stats.power.NormalIndPower.solve_power()`
>    - Input: baseline CVR, MDE, α=0.05, power=0.80
>
> 2. **Never peek at results early** — p-hacking inflates false-positive rates.
>    - Use sequential testing or adjust α if you must check early.
>
> 3. **Report effect size, not just p-value**:
>    - Cohen's d for continuous metrics
>    - Cramér's V for categorical
>    - Absolute and relative lift with confidence intervals
>
> 4. **Bootstrap is your friend** when:
>    - Sample size is small (< 1000 per group)
>    - Metric distribution is unknown
>    - You want to validate parametric test results
>
> ### 📌 Decision Framework
>
> ```
> Is the p-value < 0.05?
> ├─ YES → Is the effect size practically significant?
> │        ├─ YES → Roll out the winner
> │        └─ NO  → Consider business context (cost, risk, etc.)
> │
> └─ NO  → Is the experiment underpowered?
>          ├─ YES → Extend the experiment or accept the MDE is too small
>          └─ NO  → No effect detected — keep the control
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **P-value alone is not enough** — always report effect size and confidence intervals.
> 2. **Power analysis is mandatory** — don't run underpowered experiments.
> 3. **Bootstrap CIs** are robust and assumption-free — use them liberally.
> 4. **Multiple testing correction** (Bonferroni, FDR) is needed if you test >1 metric.
> 5. **A/A tests** validate your testing infrastructure — run them quarterly.